## trajectory-HRS

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import AgglomerativeClustering
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
from scipy.interpolate import interp1d
from scipy.signal import savgol_filter  # For smoothing
from scipy.cluster.hierarchy import dendrogram, linkage
import warnings
warnings.filterwarnings('ignore')

# Step 1: Load the data from the CSV file
df = pd.read_csv("./input/HRS_0820.csv")

# Delete NA values in 'score'
df = df.dropna(subset=['score'])

# Handle duplicates: Group by ID and wave, take mean of score if duplicates exist
df = df.groupby(['ID', 'wave']).agg({'score': 'mean'}).reset_index()

# Calculate measurement counts per ID
measurement_counts = df.groupby('ID').size()

# Filter to retain only IDs with at least 2 measurements for trajectory analysis
ids_with_at_least_2 = measurement_counts[measurement_counts >= 2].index
df = df[df['ID'].isin(ids_with_at_least_2)]

# Sort by ID and wave
df = df.sort_values(['ID', 'wave'])

# Step 2: Interpolate trajectories to a common time grid for clustering
all_times = sorted(df['wave'].unique())
min_time, max_time = df['wave'].min(), df['wave'].max()
time_grid = np.linspace(min_time, max_time, num=20)  # Increase to 20 points for better smoothing/derivative resolution

def interpolate_trajectory(group):
    if len(group) < 2:
        return np.full(len(time_grid), group['score'].values[0])
    f = interp1d(group['wave'], group['score'], kind='linear', fill_value='extrapolate')
    return f(time_grid)

unique_ids = df['ID'].unique()
interpolated_trajectories = []
smoothed_trajectories = []
derivative_features = []

for id_val in unique_ids:
    group = df[df['ID'] == id_val]
    interp_scores = interpolate_trajectory(group)
    
    # Functional Smoothing: Apply Savitzky-Golay filter (window=5, poly=2; adjust for your data smoothness)
    if len(interp_scores) >= 5:  # Window size check
        smoothed = savgol_filter(interp_scores, window_length=5, polyorder=2)
    else:
        smoothed = interp_scores  # No smoothing if too short
    
    # Derivative Feature: First-order gradient (rate of change)
    deriv = np.gradient(smoothed, time_grid)  # Amplifies variation in slopes
    
    interpolated_trajectories.append(interp_scores)
    smoothed_trajectories.append(smoothed)
    derivative_features.append(deriv)

# Combine original smoothed + derivatives to amplify variation
X_smoothed = np.array(smoothed_trajectories)
X_deriv = np.array(derivative_features)
X = np.hstack([X_smoothed, X_deriv])  # Doubles features, emphasizing dynamic variation

# Scale the enhanced data
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Step 3: Hierarchical Clustering (non-parametric)
# Compute linkage matrix for dendrogram
linkage_matrix = linkage(X_scaled, method='ward', metric='euclidean')

# Plot dendrogram to determine optimal k
plt.figure(figsize=(10, 7))
dendrogram(linkage_matrix)
plt.title('Dendrogram for Hierarchical Clustering (with Smoothing & Derivatives)')
plt.xlabel('Sample Index')
plt.ylabel('Distance')
plt.savefig('./output/dendrogram_HRS_Hierarchical_enhanced.pdf', format='pdf', bbox_inches='tight')
plt.close()

# Test k range with Silhouette (to evaluate amplification effect)
k_range = range(2, 6)
sil_scores = []
for k in k_range:
    hierarchical = AgglomerativeClustering(n_clusters=k, linkage='ward')
    labels = hierarchical.fit_predict(X_scaled)
    sil = silhouette_score(X_scaled, labels)
    sil_scores.append(sil)
    print(f"k={k}: Silhouette={sil:.2f}")

optimal_k = k_range[np.argmax(sil_scores)]
print(f"Optimal k based on max Silhouette: {optimal_k}")

# Fit with optimal k
hierarchical = AgglomerativeClustering(n_clusters=3, linkage='ward')
cluster_labels = hierarchical.fit_predict(X_scaled)

# Report cluster sizes
cluster_sizes = pd.Series(cluster_labels).value_counts()
print("Cluster sizes:\n", cluster_sizes)

# Step 4: Assign groups to original df
id_to_group = dict(zip(unique_ids, cluster_labels))
df['group'] = df['ID'].map(id_to_group)

# Output the DataFrame with group variable to CSV
df.to_csv('./output/output_with_group_HRS_Hierarchical_enhanced.csv', index=False)
print("DataFrame with group variable saved to './output/output_with_group_HRS_Hierarchical_enhanced.csv'")

# Print updated DataFrame
print("Updated DataFrame with group variable:")
print(df)

# Additional Step: Output another CSV for IDs with at least 3 measurements, computing difference between first two scores and saving the first score
ids_with_at_least_3 = measurement_counts[measurement_counts >= 3].index

diff_data = []
for id_val in ids_with_at_least_3:
    id_data = df[df['ID'] == id_val].sort_values('wave')
    if len(id_data) >= 3:
        score1 = id_data['score'].iloc[0]
        score2 = id_data['score'].iloc[1]
        score_d = score2 - score1
        diff_data.append({'ID': id_val, 'score_D': score_d, 'score1': score1})

diff_df = pd.DataFrame(diff_data)
diff_df.to_csv('./output/score_diff_HRS_Hierarchical_enhanced.csv', index=False)
print("Another CSV with score differences and first score saved to './output/score_diff_HRS_Hierarchical_enhanced.csv'")

# Step 5: Plot figures (using original score for interpretability)
# Figure 1: Mean Trajectories with Scatter and CI (The X-axis is age)
plt.figure(figsize=(12, 8))
colors = ['blue', 'green', 'red', 'purple', 'orange']  # Extended for more groups

for group_val in range(3):
    group_data = df[df['group'] == group_val]
    sns.lineplot(data=group_data, x='age', y='score', color=colors[group_val], ci=95, linewidth=2, label=f'Group {group_val} (Mean with CI)')

plt.xlabel('Age')  # Change to Age
plt.ylabel('Score')
plt.title('Mean Trajectories by Group with CI (Hierarchical Enhanced)')
plt.legend()
plt.grid(True)
plt.savefig('./output/mean_trajectories_HRS_Hierarchical_enhanced_by_age.pdf', format='pdf', bbox_inches='tight')
plt.close()

# Figure 2: Smooth Fitted Curves with Scatter and CI (The X-axis is age)
plt.figure(figsize=(12, 8))
for group_val in range(3):
    group_data = df[df['group'] == group_val]
    sns.regplot(data=group_data, x='age', y='score', order=3, ci=95, scatter=False, line_kws={'linewidth': 2}, color=colors[group_val], label=f'Group {group_val} (Smooth Fitted Curve with CI)')

plt.xlabel('Age')  # Change to Age
plt.ylabel('Score')
plt.title('Smooth Fitted Trajectories by Group with CI (Hierarchical Enhanced)')
plt.legend()
plt.grid(True)
plt.savefig('./output/smooth_fitted_trajectories_HRS_Hierarchical_enhanced_by_age.pdf', format='pdf', bbox_inches='tight')
plt.close()

# New Figure 3: Individual Trajectories (Spaghetti Plot) with Transparent Lines by Group (The X-axis is age)
plt.figure(figsize=(12, 8))
alpha_value = 0.05  # Transparency

for group_val in range(3):
    group_data = df[df['group'] == group_val]
    unique_ids_in_group = group_data['ID'].unique()
    
    for id_val in unique_ids_in_group:
        id_data = group_data[group_data['ID'] == id_val]
        plt.plot(id_data['age'], id_data['score'], color=colors[group_val], alpha=alpha_value, linewidth=1)

    # Line of mean
    sns.lineplot(data=group_data, x='age', y='score', color=colors[group_val], ci=None, linewidth=2, label=f'Group {group_val} Mean')

plt.xlabel('Age')  # Change to Age
plt.ylabel('Score')
plt.title('Individual Trajectories by Group (Transparent Lines, Hierarchical Enhanced)')
plt.legend()
plt.grid(True)
plt.savefig('./output/individual_trajectories_HRS_Hierarchical_enhanced_by_age.pdf', format='pdf', bbox_inches='tight')
plt.close()